In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

In [ ]:
df=pd.read_csv("Loan_default_noisy.csv")
df.head()

In [ ]:
print(f"Information :\n{df.info()}\nDescribe :\n{df.describe().T}\nShape :\n{df.shape}\nColumns :\n{df.columns}")

In [ ]:
print(f"NAN Values :\n{df.isna().sum()}\nDuplicate Values :{df.duplicated().sum()}")

In [ ]:
df["NumCreditLines"].value_counts()
# df['NumCreditLines'] = pd.to_numeric(df['NumCreditLines'], errors='coerce')
df["NumCreditLines"].replace("Unknown",np.nan,inplace=True)
df["NumCreditLines"].value_counts()

In [ ]:
df.info()
df["NumCreditLines"]=df["NumCreditLines"].astype("float64")
df["NumCreditLines"].dtype

In [ ]:
df.drop_duplicates(inplace=True)
df.duplicated().sum()

In [ ]:
print(df['Education'].value_counts())
print(df['Education'].unique())


df['Education'] = df['Education'].replace({'Bachelors': "Bachelor's", 'phd': 'PhD'})
print(df['Education'].unique())

In [ ]:
Q1 = df['Income'].quantile(0.25)
Q3 = df['Income'].quantile(0.75)
IQR = Q3 - Q1
df = df[(df['Income'] >= Q1 - 1.5 * IQR) & (df['Income'] <= Q3 + 1.5 * IQR)]
df = df[(df['Age'] >= 18) & (df['Age'] <= 100)]
df = df[(df['CreditScore'] >= 300) & (df['CreditScore'] <= 850)]

In [ ]:
# Index(['LoanID', 'Age', 'Income', 'LoanAmount', 'CreditScore',
#        'MonthsEmployed', 'NumCreditLines', 'InterestRate', 'LoanTerm',
#        'DTIRatio', 'Education', 'EmploymentType', 'MaritalStatus',
#        'HasMortgage', 'HasDependents', 'LoanPurpose', 'HasCoSigner',
#        'Default']
for i in df.columns:
  if df[i].dtype=="object" or i=="NumCreditLines":
    df[i].fillna(df[i].mode()[0],inplace=True)
  else:
    if i == 'Default':
      continue
    else:
      df[i].fillna(df[i].median(),inplace=True)
df.isna().sum()

In [ ]:
df=df.dropna()

In [ ]:
numeric_columns=['Age',
 'Income',
 'LoanAmount',
 'CreditScore',
 'MonthsEmployed',
 'NumCreditLines',
 'InterestRate',
 'LoanTerm',
 'DTIRatio'
 ]
categorical_columns=["Education","EmploymentType","MaritalStatus","HasMortgage","HasDependents","LoanPurpose","HasCoSigner"]

In [ ]:
for i in categorical_columns:
    print(i)
    print(df[i].value_counts().sort_values(ascending=False))
    plt.figure(figsize=(6,4))
    plt.title(i)
    sns.countplot(x=df[i],hue=df["Default"])
    plt.show()

In [ ]:
for i in numeric_columns:
    plt.figure(figsize=(10,5))
    plt.title(i)
    sns.histplot(x=df[i],kde=True,hue=df["Default"])
    plt.show()

In [ ]:
sns.pairplot(df,hue="Default")

In [ ]:
for i in numeric_columns:
  for j in categorical_columns:
    sns.barplot(x=df[j],y=df[i],hue=df["Default"])
    plt.show()

In [ ]:
for i in numeric_columns:
  for j in categorical_columns:
    sns.boxplot(y=df[j],x=df[i],hue=df["Default"])
    plt.show()

In [ ]:
# Correlation Heatmap
plt.figure(figsize=(10,8))
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap='coolwarm')
plt.show()

df1=pd.DataFrame(df[df["Default"]==1])
df0=pd.DataFrame(df[df["Default"]==0])


plt.figure(figsize=(10,8))
plt.title("Correlation Heatmap for Defaulters")
sns.heatmap(df1.corr(numeric_only=True), annot=True, cmap='coolwarm')
plt.show()

plt.figure(figsize=(10,8))
plt.title("Correlation Heatmap for Non Defaulters")
sns.heatmap(df0.corr(numeric_only=True), annot=True, cmap='coolwarm')
plt.show()

In [ ]:
print(df["MaritalStatus"].unique(),
df["EmploymentType"].unique())

In [ ]:
df=pd.get_dummies(df,columns=["Education","EmploymentType","MaritalStatus","LoanPurpose"],drop_first=True)

In [ ]:
df.head()

In [ ]:
df.drop("LoanID",axis=1,inplace=True)

In [ ]:
# df=df.astype("int")
df.info()

In [ ]:
print([df[i].unique() for i in ["HasMortgage","HasDependents","HasCoSigner"]])

In [ ]:
df[["HasMortgage", "HasDependents", "HasCoSigner"]] = df[["HasMortgage", "HasDependents", "HasCoSigner"]].replace({'Yes': 1, 'No': 0})

In [ ]:
df=df.astype("float64")

In [ ]:
df

In [ ]:
df["Loan_to_Income_Ratio"]=df['LoanAmount'] / df['Income']

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score,confusion_matrix,classification_report,f1_score
# from sklearn.svm import SupportMachineClassifier
from sklearn.neighbors import KNeighborsClassifier

In [ ]:
y=df["Default"]
x_ns=df.drop("Default",axis=1)

In [ ]:
y.value_counts()

In [ ]:
x_train,x_test,y_train,y_test=train_test_split(x_ns,y,test_size=0.1,random_state=42)

In [ ]:
ss=StandardScaler()
x_train_scaled=ss.fit_transform(x_train)
x_test_scaled=ss.transform(x_test)

In [ ]:
from imblearn.over_sampling import SMOTE
sm = SMOTE(random_state=42)
print(f"Before SMOTE: {y_train.value_counts()}")
x_train_scaled, y_train = sm.fit_resample(x_train_scaled, y_train)
print(f"After SMOTE: {y_train.value_counts()}")

In [ ]:
from sklearn.decomposition import PCA
pca = PCA(n_components=0.95)
x_train_pca = pca.fit_transform(x_train_scaled)
x_test_pca = pca.transform(x_test_scaled)

In [ ]:
model_lr=LogisticRegression(class_weight='balanced')
model_lr.fit(x_train_pca,y_train)

In [ ]:
y_pred=model_lr.predict(x_test_pca)
print(f1_score(y_test,y_pred,average='weighted'))

print(f"accracy score : {accuracy_score(y_test,y_pred)}\nconfusion matrix : \n{confusion_matrix(y_test,y_pred)}\nclassification report : \n{classification_report(y_test,y_pred)}")

In [ ]:
# model_lr=LogisticRegression(class_weight={0:1,1:8})
# model_lr=LogisticRegression(class_weight="balanced")
# model_lr.fit(x_train_scaled,y_train)

In [ ]:
# y_pred=model_lr.predict(x_test_scaled)
# print(f"accracy score : {accuracy_score(y_test,y_pred)}\nconfusion matrix : \n{confusion_matrix(y_test,y_pred)}\nclassification report : \n{classification_report(y_test,y_pred)}")

In [ ]:
# model_knn=KNeighborsClassifier(n_neighbors=5,weights='distance')
model_knn=KNeighborsClassifier(n_neighbors=5)
model_knn.fit(x_train_pca,y_train)

In [ ]:
y_pred=model_knn.predict(x_test_pca)
print(f"f1 Score :{f1_score(y_test,y_pred,average='weighted')}")

print(f"accracy score : {accuracy_score(y_test,y_pred)}\nconfusion matrix : \n{confusion_matrix(y_test,y_pred)}\nclassification report : \n{classification_report(y_test,y_pred)}")

In [ ]:
from sklearn.tree import DecisionTreeClassifier
model_dt=DecisionTreeClassifier(
    class_weight='balanced',
    max_depth=10,
    min_samples_leaf=20)
model_dt.fit(x_train_pca,y_train)

In [ ]:
y_pred=model_dt.predict(x_test_pca)
print(f"f1 Score :{f1_score(y_test,y_pred,average='weighted')}")
print(f"accracy score : {accuracy_score(y_test,y_pred)}\nconfusion matrix : \n{confusion_matrix(y_test,y_pred)}\nclassification report : \n{classification_report(y_test,y_pred)}")

In [ ]:
from sklearn.svm import LinearSVC

model_fast_svm = LinearSVC(class_weight='balanced', random_state=42, max_iter=2000)
model_fast_svm.fit(x_train_pca, y_train)

In [ ]:
y_pred=model_fast_svm.predict(x_test_pca)
print(f"f1 Score :{f1_score(y_test,y_pred,average='weighted')}")
print(f"accracy score : {accuracy_score(y_test,y_pred)}\nconfusion matrix : \n{confusion_matrix(y_test,y_pred)}\nclassification report : \n{classification_report(y_test,y_pred)}")

In [ ]:
from sklearn.svm import SVC
# model_svm=SVC(kernel="linear")
model_svm=SVC(kernel="rbf",class_weight='balanced')
model_svm.fit(x_train_pca[:30000],y_train[:30000])

In [ ]:
y_pred=model_svm.predict(x_test_pca)

print(f"accracy score : {accuracy_score(y_test,y_pred)}\nconfusion matrix : \n{confusion_matrix(y_test,y_pred)}\nclassification report : \n{classification_report(y_test,y_pred)}")
print(f"F1 Score :{f1_score(y_test,y_pred,average='weighted')}")

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score


model_rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=12,
    class_weight='balanced_subsample',
    random_state=42,
    n_jobs=-1
)


model_rf.fit(x_train_pca, y_train)


y_pred = model_rf.predict(x_test_pca)

In [ ]:
y_pred=model_rf.predict(x_test_pca)

print(f"accracy score : {accuracy_score(y_test,y_pred)}\nconfusion matrix : \n{confusion_matrix(y_test,y_pred)}\nclassification report : \n{classification_report(y_test,y_pred)}")
print(f"F1 Score :{f1_score(y_test,y_pred,average='weighted')}")

In [ ]:

y_probs = model_rf.predict_proba(x_test_pca)[:, 1]

#
y_pred_custom = (y_probs >= 0.47).astype(int)

print(f"Final Accuracy: {accuracy_score(y_test, y_pred_custom):.4}\nconfusion matrix : \n{confusion_matrix(y_test,y_pred_custom)}")
print("\nFinal Classification Report:\n", classification_report(y_test, y_pred_custom))

In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix


scale_weight = float(y_train.value_counts()[0] / y_train.value_counts()[1])


xgb_model = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=scale_weight,
    use_label_encoder=False,
    eval_metric='logloss'
)


xgb_model.fit(x_train_pca, y_train)

y_pred = xgb_model.predict(x_test_pca)


print(f"accracy score : {accuracy_score(y_test,y_pred)}\nconfusion matrix : \n{confusion_matrix(y_test,y_pred)}\nclassification report : \n{classification_report(y_test,y_pred)}")
print(f"F1 Score :{f1_score(y_test,y_pred,average='weighted')}")

In [ ]:
from sklearn.ensemble import VotingClassifier


voting_clf = VotingClassifier(
    estimators=[
        ('lr', model_lr),    # High Recall
        ('rf', model_rf),    # Stability
        ('xgb', xgb_model)   # High F1-Score
    ],
    # final_estimator=LogisticRegression(class_weight='balanced'),
    voting='soft'
)

voting_clf.fit(x_train_pca, y_train)

In [ ]:
y_pred = voting_clf.predict(x_test_pca)


print(f"accracy score : {accuracy_score(y_test,y_pred)}\nconfusion matrix : \n{confusion_matrix(y_test,y_pred)}\nclassification report : \n{classification_report(y_test,y_pred)}")
print(f"F1 Score :{f1_score(y_test,y_pred,average='weighted')}")